In [7]:
import operator
import os


from dotenv import load_dotenv
import httpx

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from typing import Literal, TypedDict, Annotated, List
import uuid
import base64
import ssl
from langgraph.graph import StateGraph, START, END
from langchain.tools import tool

load_dotenv()
ssl_cert_file=""
client= httpx.Client(
        verify=ssl.create_default_context(cafile=ssl_cert_file)
    )

client_id=os.environ.get('LLM_CLIENT_ID')
client_secret=os.environ.get('LLM_CLIENT_SECRET')



auth_string = f"{client_id}:{client_secret}"
auth_header = f"Basic {base64.b64encode(auth_string.encode()).decode()}"

llm_openai = ChatOpenAI()


@tool
def multiply(a:int,b:int)->int:
   """
    This tool is used for multiplication of two numbers.
   """
   return a*b

'''
add: used for addition of 2 number
'''
@tool
def add(a:int,b:int)->int:
    """
    This tool is used for addition of two numbers.
   """
    return a+b

llm_with_tools=llm_openai.bind_tools([multiply,add])

def run_agent(question: str):
    messages = [HumanMessage(content=question)]
    tools_map = {"multiply": multiply, "add": add}
    while True:
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        print(f"\n[AIMessage]", end=" ")
        if response.tool_calls:
            calls = [f"{t['name']}({t['args']})" for t in response.tool_calls]
            print(f"tool_calls: {calls}")
        else:
            print(response.content)

        if not response.tool_calls:
            return response.content

        for tool_call in response.tool_calls:
            actual_tool = tools_map[tool_call["name"]]
            result = actual_tool.invoke(tool_call["args"])
            print(f"[ToolMessage] {tool_call['name']} returned: {result}")
            messages.append(ToolMessage(content=str(result), tool_call_id=tool_call["id"]))
result=run_agent("What is (15 + 7) * 3?")
print(result)


[AIMessage] tool_calls: ["add({'a': 15, 'b': 7})"]
[ToolMessage] add returned: 22

[AIMessage] tool_calls: ["multiply({'a': 22, 'b': 3})"]
[ToolMessage] multiply returned: 66

[AIMessage] The result of \((15 + 7) \times 3\) is **66**.
The result of \((15 + 7) \times 3\) is **66**.


In [15]:
import operator
import os


from dotenv import load_dotenv
import httpx

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from typing import Literal, TypedDict, Annotated, List
import uuid
import base64
import ssl
from langgraph.graph import StateGraph, START, END, add_messages
from langchain.tools import tool

load_dotenv()
ssl_cert_file=""
client= httpx.Client(
        verify=ssl.create_default_context(cafile=ssl_cert_file)
    )

client_id=os.environ.get('LLM_CLIENT_ID')
client_secret=os.environ.get('LLM_CLIENT_SECRET')



auth_string = f"{client_id}:{client_secret}"
auth_header = f"Basic {base64.b64encode(auth_string.encode()).decode()}"

llm_openai = ChatOpenAI()


@tool
def multiply(a:int,b:int)->int:
   """
    This tool is used for multiplication of two numbers.
   """
   return a*b

'''
add: used for addition of 2 number
'''
@tool
def add(a:int,b:int)->int:
    """
    This tool is used for addition of two numbers.
   """
    return a+b

llm_with_tools=llm_openai.bind_tools([multiply,add])

class AgentState(TypedDict):
    messages: Annotated[List, add_messages]

def agent_node(state:AgentState):
    response = llm_with_tools.invoke(state["messages"])   # pass ALL messages
    return {"messages": [response]}

def tool_node(state:AgentState):
    response = state["messages"][-1]
    tools_map = {"multiply": multiply, "add": add}
    messages=[]
    for tool_call in response.tool_calls:
        actual_tool = tools_map[tool_call["name"]]
        result = actual_tool.invoke(tool_call["args"])
        messages.append(ToolMessage(content=str(result), tool_call_id=tool_call["id"]))
    return {"messages":messages}

def conditional_node(state:AgentState):
    response = state["messages"][-1]
    if response.tool_calls:
        return "tool_node"
    return "end"

f_graph=StateGraph(AgentState)
f_graph.add_node("agent_node",agent_node)
f_graph.add_node("tool_node",tool_node)

f_graph.add_edge(START,"agent_node")
f_graph.add_conditional_edges("agent_node",conditional_node,{"tool_node":"tool_node","end":END})
f_graph.add_edge("tool_node","agent_node")

app=f_graph.compile()

result=app.invoke({"messages": [HumanMessage(content="What is (15 + 7) * 3?")]})

print(result)

print(result)

{'messages': [HumanMessage(content='What is (15 + 7) * 3?', additional_kwargs={}, response_metadata={}, id='914fd613-93b5-48cb-b70a-bf265d4fd799'), AIMessage(content='', additional_kwargs={}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'gpt-oss-120b', 'system_fingerprint': 'vllm-0.30.0-tp4-81e3a87a', 'model_provider': 'openai'}, id='lc_run--01a11b51-d025-7f23-925d-7d9c0bd22324', tool_calls=[{'name': 'add', 'args': {'a': 15, 'b': 7}, 'id': 'chatcmpl-tool-be2ed3c905a7160d', 'type': 'tool_call'}], invalid_tool_calls=[]), ToolMessage(content='22', id='eb20ab70-6370-4f69-965b-719f2c543e7d', tool_call_id='chatcmpl-tool-be2ed3c905a7160d'), AIMessage(content='', additional_kwargs={}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'gpt-oss-120b', 'system_fingerprint': 'vllm-0.30.0-tp4-81e3a87a', 'model_provider': 'openai'}, id='lc_run--01a11b51-d7c5-7c21-9e1f-3e9281a6bde0', tool_calls=[{'name': 'multiply', 'args': {'a': 22, 'b': 3}, 'id': 'chatcmpl-tool-9d99

In [18]:
from langgraph.checkpoint.memory import MemorySaver
import operator
import os


from dotenv import load_dotenv
import httpx

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from typing import Literal, TypedDict, Annotated, List
import uuid
import base64
import ssl
from langgraph.graph import StateGraph, START, END, add_messages
from langchain.tools import tool

load_dotenv()
ssl_cert_file=""
client= httpx.Client(
        verify=ssl.create_default_context(cafile=ssl_cert_file)
    )

client_id=os.environ.get('LLM_CLIENT_ID')
client_secret=os.environ.get('LLM_CLIENT_SECRET')



auth_string = f"{client_id}:{client_secret}"
auth_header = f"Basic {base64.b64encode(auth_string.encode()).decode()}"

llm_openai = ChatOpenAI()


@tool
def multiply(a:int,b:int)->int:
   """
    This tool is used for multiplication of two numbers.
   """
   return a*b

'''
add: used for addition of 2 number
'''
@tool
def add(a:int,b:int)->int:
    """
    This tool is used for addition of two numbers.
   """
    return a+b

llm_with_tools=llm_openai.bind_tools([multiply,add])

class AgentState(TypedDict):
    messages: Annotated[List, add_messages]

def agent_node(state:AgentState):
    response = llm_with_tools.invoke(state["messages"])   # pass ALL messages
    return {"messages": [response]}

def tool_node(state:AgentState):
    response = state["messages"][-1]
    tools_map = {"multiply": multiply, "add": add}
    messages=[]
    for tool_call in response.tool_calls:
        actual_tool = tools_map[tool_call["name"]]
        result = actual_tool.invoke(tool_call["args"])
        messages.append(ToolMessage(content=str(result), tool_call_id=tool_call["id"]))
    return {"messages":messages}

def conditional_node(state:AgentState):
    response = state["messages"][-1]
    if response.tool_calls:
        return "tool_node"
    return "end"

f_graph=StateGraph(AgentState)
f_graph.add_node("agent_node",agent_node)
f_graph.add_node("tool_node",tool_node)

f_graph.add_edge(START,"agent_node")
f_graph.add_conditional_edges("agent_node",conditional_node,{"tool_node":"tool_node","end":END})
f_graph.add_edge("tool_node","agent_node")
memory = MemorySaver()
app=f_graph.compile(checkpointer=memory,interrupt_before=["tool_node"])
config = {"configurable": {"thread_id": "session_1"}}
result=app.invoke({"messages": [HumanMessage(content="What is (15 + 7) * 3?")]},config=config)

print(result)

state = app.get_state(config)
last_message = state.values["messages"][-1]
print("Agent wants to call:", last_message.tool_calls)
# Human reviews here — approve or reject
result=app.invoke(None,config=config)
print(result["messages"][-1].content)
print("\n================")

state = app.get_state(config)
last_message = state.values["messages"][-1]
print("Agent wants to call1:", last_message.tool_calls)
# Human reviews here — approve or reject
result=app.invoke(None,config=config)
print(result["messages"][-1].content)
print("\n================")


{'messages': [HumanMessage(content='What is (15 + 7) * 3?', additional_kwargs={}, response_metadata={}, id='aabb6300-f11c-45c2-978d-6a44b45fc474'), AIMessage(content='', additional_kwargs={}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'gpt-oss-120b', 'system_fingerprint': 'vllm-0.30.0-tp4-81e3a87a', 'model_provider': 'openai'}, id='lc_run--01a11b6d-8d22-7f92-b606-07ac12757a47', tool_calls=[{'name': 'add', 'args': {'a': 15, 'b': 7}, 'id': 'chatcmpl-tool-b7d261eddb724315', 'type': 'tool_call'}], invalid_tool_calls=[])]}
Agent wants to call: [{'name': 'add', 'args': {'a': 15, 'b': 7}, 'id': 'chatcmpl-tool-b7d261eddb724315', 'type': 'tool_call'}]


Agent wants to call1: [{'name': 'multiply', 'args': {'a': 22, 'b': 3}, 'id': 'chatcmpl-tool-bb6a7f3231ccbe0d', 'type': 'tool_call'}]
The result is **66**.

